# nb55 - Remaining-arsenal sweep I: Muon, capacity, training length (H19)

**Context.** Constraint update 2026-08-04: no additional data is coming - aggregate 0.030 must come from the existing sample alone. This notebook opens the systematic sweep of every remaining untested lever, one variable per config on the adopted EMA recipe.

**Hypotheses (each vs the EMA anchor 0.0424 +/- 0.0003, all hyperparameters fixed a priori, no scans):**
- H19a `muon`: Muon optimizer (Newton-Schulz orthogonalized momentum) on 2-D weights + AdamW on the rest - tabular evidence beats AdamW at small model scale (arXiv:2604.15297). lr_muon 0.02, momentum 0.95.
- H19b `d256`: double the width (d=128 -> 256, ~4x params). The nb22 capacity scan predates quant, clean-aux and EMA; EMA's variance reduction may let the larger model train where it previously overfit.
- H19c `ep300`: 300-epoch cosine with patience 40 - EMA at decay 0.999 has a ~1000-step memory and benefits from longer schedules.

**Proof criterion.** 2 seeds each; win = >0.002 overall or in any E>17 bin vs 0.0424 +/- 0.0003. Anything within noise is falsified and recorded.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution, PITCH, EPS
from picocal_data import build_grid, prep
from picocal_models import SubNetFQ, QUANTILES, width_binned_calibration, CFG
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB55_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB55_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
D = prep(4, ME, CE, ng=6)
T = dict(X=torch.from_numpy(D['X']).to(DEVICE), M=torch.from_numpy(D['M']).to(DEVICE),
         G=torch.from_numpy(D['G']).to(DEVICE), Y=torch.from_numpy(D['y']).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(D['Eraw']).to(DEVICE))
ktr, kva, kte, ctr = D['ktr'], D['kva'], D['kte'], D['ctr']
y = D['y']; Et = D['Et']
QS = torch.tensor(QUANTILES, device=DEVICE)
print(f'device {DEVICE} | mode {MODE} | build+prep {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events


W=4: N 102857 (main 72554 + aux 30303), tr/va/te 50787/10883/10884, IN_DIM 16
device cuda | mode full | build+prep 159s


In [2]:
def ns5(G, steps=5):
    a, b, cc = 3.4445, -4.7750, 2.0315
    X = G.float()
    tall = G.size(-2) > G.size(-1)
    if tall: X = X.mT
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        A = X @ X.mT
        B = b * A + cc * (A @ A)
        X = a * X + B @ X
    if tall: X = X.mT
    return X.to(G.dtype)
class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95):
        super().__init__(params, dict(lr=lr, momentum=momentum))
    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                st = self.state[p]
                if 'buf' not in st: st['buf'] = torch.zeros_like(p.grad)
                buf = st['buf']
                buf.mul_(group['momentum']).add_(p.grad)
                g = p.grad.add(buf, alpha=group['momentum'])
                if g.ndim == 2:
                    g = ns5(g)
                    scale = max(1.0, p.size(0) / p.size(1)) ** 0.5
                else:
                    scale = 1.0
                p.add_(g, alpha=-group['lr'] * scale)
CONFS = dict(
    muon=dict(d=CFG['d'], epochs_full=100, patience_full=15, opt='muon'),
    d256=dict(d=256, epochs_full=100, patience_full=15, opt='adamw'),
    ep300=dict(d=CFG['d'], epochs_full=300, patience_full=40, opt='adamw'))
def train_eval(config, seed):
    cf = CONFS[config]
    epochs = 2 if MODE == 'smoke' else cf['epochs_full']
    patience = 99 if MODE == 'smoke' else cf['patience_full']
    cfg2 = dict(CFG); cfg2['d'] = cf['d']
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6, cfg=cfg2).to(DEVICE)
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
    if cf['opt'] == 'muon':
        p2 = [p for p in model.parameters() if p.ndim == 2]
        p1 = [p for p in model.parameters() if p.ndim != 2]
        opts = [Muon(p2, lr=0.02, momentum=0.95),
                torch.optim.AdamW(p1, lr=CFG['lr'], weight_decay=CFG['wd'])]
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opts[1], T_max=epochs)
    else:
        opts = [torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])]
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opts[0], T_max=epochs)
    tr_idx = np.concatenate([np.asarray(ktr), ctr])
    ck = CKPT / f'nb55_{config}_s{seed}.pt'
    def batches(idx, bs, sh=None):
        idx = np.asarray(idx)
        if sh is not None: idx = sh.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(m, b): return m(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def vloss(m):
        m.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256):
                d = T['Y'][b] - fwd(m, b)
                s += torch.maximum(QS * d, (QS - 1) * d).mean().item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); ema.load_state_dict(st['ema'])
        for o, so in zip(opts, st['opts']): o.load_state_dict(so)
        sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume {config} s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(tr_idx, CFG['batch'], rng):
            for o in opts: o.zero_grad()
            d = T['Y'][b] - fwd(model, b)
            torch.maximum(QS * d, (QS - 1) * d).mean().backward()
            for o in opts: o.step()
            ema.update_parameters(model)
        sched.step()
        vv = vloss(ema.module)
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(ema.module.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), ema=ema.state_dict(), opts=[o.state_dict() for o in opts],
                        sched=sched.state_dict(), best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    final = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6, cfg=cfg2).to(DEVICE)
    final.load_state_dict(bstate); final.eval()
    def run(idx):
        out = []
        with torch.no_grad():
            for b in batches(idx, 256): out.append(fwd(final, b).cpu().numpy())
        return np.concatenate(out)
    pe = width_binned_calibration(run(kva), run(kte), y[kva])
    return float(resolution(pe, Et[kte])['sigma_eff']), pe

In [3]:
JOBS = {'smoke': [('muon', 0), ('d256', 0), ('ep300', 0)],
        'full': [(cfg, s) for cfg in ('muon', 'd256', 'ep300') for s in (0, 1)]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb55_sweep{TAG}.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['config'], prev['seed']))
    print('resume, done:', sorted(done))
for config, seed in JOBS:
    if (config, seed) in done: print('skip', config, seed); continue
    t1 = time.time()
    sig, pe = train_eval(config, seed)
    np.save(OUT / f'nb55_pred{TAG}_{config}_s{seed}.npy', pe)
    row = dict(config=config, seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'{config} seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

muon seed 0: sigma_eff 0.1426 (1929s)


muon seed 1: sigma_eff 0.1448 (611s)


d256 seed 0: sigma_eff 0.0422 (4240s)


d256 seed 1: sigma_eff 0.0430 (4808s)


ep300 seed 0: sigma_eff 0.0426 (2620s)


ep300 seed 1: sigma_eff 0.0417 (2215s)


config  seed  sigma_eff  elapsed
  muon     0     0.1426     1929
  muon     1     0.1448      611
  d256     0     0.0422     4240
  d256     1     0.0430     4808
 ep300     0     0.0426     2620
 ep300     1     0.0417     2215


## Verdict vs the EMA anchor

Anchor 0.0424 +/- 0.0003 (nb52 EMA singles). Win = >0.002 overall or in any E>17 bin; per-config per-bin below.

In [4]:
te_e = Et[kte]
edges = np.quantile(te_e, np.linspace(0, 1, 7))
def perbin(pe):
    out = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        out.append(resolution(pe[mm], te_e[mm])['sigma_eff'])
    return out
print('anchor: EMA singles 0.0424 +/- 0.0003 | stack record 0.0409')
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
for cfg in ('muon', 'd256', 'ep300'):
    preds = [np.load(OUT / f'nb55_pred{TAG}_{cfg}_s{s}.npy') for s in SEEDS
             if (OUT / f'nb55_pred{TAG}_{cfg}_s{s}.npy').exists()]
    if not preds: continue
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    print(f'{cfg:6s} mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}')
    print('       per-bin ' + ' / '.join(f'{b:.4f}' for b in perbin(ens)))

anchor: EMA singles 0.0424 +/- 0.0003 | stack record 0.0409
muon   mean 0.1437 +/- 0.0011 | ens 0.1413
       per-bin 0.1867 / 0.1571 / 0.1233 / 0.1156 / 0.1254 / 0.1219
d256   mean 0.0426 +/- 0.0004 | ens 0.0421


       per-bin 0.0644 / 0.0472 / 0.0370 / 0.0360 / 0.0342 / 0.0357
ep300  mean 0.0421 +/- 0.0004 | ens 0.0417
       per-bin 0.0650 / 0.0470 / 0.0353 / 0.0349 / 0.0345 / 0.0352
